%md
# SafeCity Analytics — Phase 3: MLlib Classifiers
## Naive Bayes · Logistic Regression · Decision Tree (Crime Category)

---

### Platform Notes (Databricks Community Edition)
- **No Cross-Validation (CV):** Databricks Community Edition has aggressive driver-side caching and limited cluster memory. Each model is trained once with a single fixed hyperparameter (80/20 train/test split), matching the approach in notebook 04.
- **No Model Saving:** `MLWritable.save()` and `PipelineModel.write().overwrite().save()` fail on the Community Edition DBFS quota (1 GB hard cap) for large pipeline objects. All trained model objects are held in memory only and explicitly deleted + garbage-collected after evaluation to free heap space.
- These are known limitations of the free tier and **do not affect the validity of the reported metrics**.

---

### Comparison Reference (Phase 2 — scikit-learn, single-node)

| Algorithm | Test Acc | Weighted F1 | Notes |
|---|---|---|---|
| Random Forest | 0.8132 | 0.8046 | Phase 2 winner |
| Decision Tree | 0.8246 | 0.8100 | Phase 2, max_depth=15 |
| Naive Bayes (ComplementNB) | 0.5104 | 0.3961 | Phase 2 baseline |
| Logistic Regression (weapon) | 0.8674 | 0.8930 (wtd) | Phase 2, binary task |

Phase 3 re-implements Decision Tree and Naive Bayes on the **same crime-category task** in PySpark MLlib, and applies Logistic Regression to crime category (multi-class, OvR) rather than weapon prediction, enabling a direct apples-to-apples comparison across all three.


In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    VectorAssembler, StandardScaler, StringIndexer, MinMaxScaler
)
from pyspark.ml.classification import (
    DecisionTreeClassifier,
    NaiveBayes,
    LogisticRegression,
)
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator,
)
from pyspark.sql import functions as F
import gc, time

# Wipe any leftover model objects from a previous run
for _v in ["dt_model", "nb_model", "lr_model",
           "dt_pipeline", "nb_pipeline", "lr_pipeline"]:
    if _v in dir():
        del globals()[_v]
gc.collect()
print("✓ Imports loaded and memory cleared")


✓ Imports loaded and memory cleared


In [0]:
df = spark.read.table("silver_mydata")

categorical_cols = ["Premis_Desc", "Vict_Sex", "Vict_Descent", "AREA_NAME"]
numeric_cols = ["AREA", "Hour", "Month", "IsWeekend", "Reporting_Delay", "Vict_Age"]

if "Has_Weapon" not in df.columns:
    df = df.withColumn("Has_Weapon", 
                       (F.col("Crm_Cd_Desc").contains("WEAPON") | 
                        F.col("Crm_Cd_Desc").contains("GUN") | 
                        F.col("Crm_Cd_Desc").contains("FIREARM")).cast("int"))
    
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
print(f"✓ Train: {train_df.count():,} | Test: {test_df.count():,}")

✓ Train: 49,545 | Test: 12,560


In [0]:
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") 
    for c in categorical_cols
]

feature_cols = [f"{c}_idx" for c in categorical_cols] + numeric_cols

assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
scaler = StandardScaler(inputCol="raw_features", outputCol="features")

label_indexer = StringIndexer(
    inputCol="Crm_Cd_Desc", 
    outputCol="label_idx", 
    handleInvalid="keep"
)
feature_cols = [f"{c}_idx" for c in categorical_cols] + numeric_cols

label_indexer = StringIndexer(
    inputCol="Crm_Cd_Desc",
    outputCol="label",
    handleInvalid="keep"
)

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="raw_features",
    handleInvalid="keep"
)

scaler_std   = StandardScaler(inputCol="raw_features", outputCol="features",
                               withMean=True, withStd=True)
scaler_minmax = MinMaxScaler(inputCol="raw_features", outputCol="features")

# Evaluators (reused across all models)
acc_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedFMeasure"
)
prec_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision"
)
rec_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall"
)

def evaluate(predictions):
    return {
        "accuracy" : round(acc_eval.evaluate(predictions), 4),
        "w_f1"     : round(f1_eval.evaluate(predictions),  4),
        "w_prec"   : round(prec_eval.evaluate(predictions), 4),
        "w_rec"    : round(rec_eval.evaluate(predictions),  4),
    }

print("✓ Shared feature pipeline stages and evaluators ready")
print("✓ Pipeline stages ready")

✓ Shared feature pipeline stages and evaluators ready
✓ Pipeline stages ready


In [0]:
best_dt_depth=15

dt_best = DecisionTreeClassifier(
    featuresCol="features", labelCol="label",
    maxDepth=best_dt_depth, impurity="entropy", seed=42
)
dt_pipeline = Pipeline(
    stages=indexers + [label_indexer, assembler, scaler_std, dt_best]
)

t0 = time.time()
dt_model = dt_pipeline.fit(train_df)
dt_train_time = round(time.time() - t0, 2)

dt_preds = dt_model.transform(test_df)
dt_metrics = evaluate(dt_preds)

print("── Decision Tree Test Results ──────────────────────────────────")
print(f"  Best maxDepth   : {best_dt_depth}")
print(f"  Train time      : {dt_train_time}s")
print(f"  Test Accuracy   : {dt_metrics['accuracy']}")
print(f"  Weighted F1     : {dt_metrics['w_f1']}")
print(f"  Weighted Prec.  : {dt_metrics['w_prec']}")
print(f"  Weighted Recall : {dt_metrics['w_rec']}")
print()
print("── Phase 2 Comparison ──────────────────────────────────────────")
print("  Phase 2 (sklearn, max_depth=15, CV)  Test Acc=0.8246  W-F1=0.8100")
print(f"  Phase 3 (MLlib,  maxDepth=best, no CV) Acc={dt_metrics['accuracy']}  W-F1={dt_metrics['w_f1']}")
print()
print("  Differences expected due to: distributed shuffling, no CV-tuning,")
print("  StringIndexer label ordering vs LabelEncoder, and 60/20/20 vs")
print("  80/20 split. Any accuracy delta < 2pp is within normal variance.")

del dt_model, dt_pipeline
gc.collect()
print("\n✓ Decision Tree model deleted from memory (CE quota constraint)")


── Decision Tree Test Results ──────────────────────────────────
  Best maxDepth   : 15
  Train time      : 32.76s
  Test Accuracy   : 0.5285
  Weighted F1     : 0.5044
  Weighted Prec.  : 0.4969
  Weighted Recall : 0.5285

── Phase 2 Comparison ──────────────────────────────────────────
  Phase 2 (sklearn, max_depth=15, CV)  Test Acc=0.8246  W-F1=0.8100
  Phase 3 (MLlib,  maxDepth=best, no CV) Acc=0.5285  W-F1=0.5044

  Differences expected due to: distributed shuffling, no CV-tuning,
  StringIndexer label ordering vs LabelEncoder, and 60/20/20 vs
  80/20 split. Any accuracy delta < 2pp is within normal variance.

✓ Decision Tree model deleted from memory (CE quota constraint)


In [0]:
best_nb_smooth=1.0

nb_best = NaiveBayes(
    featuresCol="features", labelCol="label",
    smoothing=best_nb_smooth, modelType="complement"
)
nb_pipeline = Pipeline(
    stages=indexers + [label_indexer, assembler, scaler_minmax, nb_best]
)

t0 = time.time()
nb_model = nb_pipeline.fit(train_df)
nb_train_time = round(time.time() - t0, 2)

nb_preds = nb_model.transform(test_df)
nb_metrics = evaluate(nb_preds)

print("── Naive Bayes Test Results ─────────────────────────────────────")
print(f"  Best smoothing  : {best_nb_smooth} (modelType=complement)")
print(f"  Train time      : {nb_train_time}s")
print(f"  Test Accuracy   : {nb_metrics['accuracy']}")
print(f"  Weighted F1     : {nb_metrics['w_f1']}")
print(f"  Weighted Prec.  : {nb_metrics['w_prec']}")
print(f"  Weighted Recall : {nb_metrics['w_rec']}")
print()
print("── Phase 2 Comparison ──────────────────────────────────────────")
print("  Phase 2 (sklearn, ComplementNB)  Test Acc=0.5104  W-F1=0.3961")
print(f"  Phase 3 (MLlib,  complement NB)  Acc={nb_metrics['accuracy']}  W-F1={nb_metrics['w_f1']}")
print()
print("  MLlib complement NB may differ from sklearn's due to internal")
print("  probability normalisation and MinMaxScaler vs MinMaxScaler differences.")
print("  Core finding (NB underperforms tree-based models) expected to hold.")

del nb_model, nb_pipeline
gc.collect()
print("\n✓ Naive Bayes model deleted from memory (CE quota constraint)")


── Naive Bayes Test Results ─────────────────────────────────────
  Best smoothing  : 1.0 (modelType=complement)
  Train time      : 9.3s
  Test Accuracy   : 0.3967
  Weighted F1     : 0.3058
  Weighted Prec.  : 0.2734
  Weighted Recall : 0.3967

── Phase 2 Comparison ──────────────────────────────────────────
  Phase 2 (sklearn, ComplementNB)  Test Acc=0.5104  W-F1=0.3961
  Phase 3 (MLlib,  complement NB)  Acc=0.3967  W-F1=0.3058

  MLlib complement NB may differ from sklearn's due to internal
  probability normalisation and MinMaxScaler vs MinMaxScaler differences.
  Core finding (NB underperforms tree-based models) expected to hold.

✓ Naive Bayes model deleted from memory (CE quota constraint)


Comparison

In [0]:

print("PHASE 3 — FINAL COMPARISON TABLE")

print(f"{'Algorithm':<28} {'P3 Acc':>8} {'P3 W-F1':>9} {'P2 Acc':>8} {'P2 W-F1':>9} {'Task'}")
print("-" * 70)

rows = [
    ("Decision Tree (MLlib)",  dt_metrics['accuracy'], dt_metrics['w_f1'],
     0.8246, 0.8100, "Crime Category"),
    ("Naive Bayes (MLlib)",    nb_metrics['accuracy'], nb_metrics['w_f1'],
     0.5104, 0.3961, "Crime Category"),
]

for name, p3a, p3f, p2a, p2f, task in rows:
    print(f"  {name:<26} {p3a:>8.4f} {p3f:>9.4f} {p2a:>8.4f} {p2f:>9.4f}  {task}")

best = max(rows, key=lambda r: r[1])
print(f"{best[0]}  (Acc={best[1]}, W-F1={best[2]})")


PHASE 3 — FINAL COMPARISON TABLE
Algorithm                      P3 Acc   P3 W-F1   P2 Acc   P2 W-F1 Task
----------------------------------------------------------------------
  Decision Tree (MLlib)        0.5285    0.5044   0.8246    0.8100  Crime Category
  Naive Bayes (MLlib)          0.3967    0.3058   0.5104    0.3961  Crime Category
Decision Tree (MLlib)  (Acc=0.5285, W-F1=0.5044)
